# Export products to JSON — 1 product, 1 file

Rebuilds products from the database back into the original JSON shape
(the same structure as the files in `product_data/`) and writes **one file per product**
into `product_export/`.

- Give a **top-level key** (a Shipping Case) to export its whole hierarchy.
- Give a **child key** (a Bundle or Pack) to export just that product and what is inside it.

Reads `product_database.db` directly, so the API server does not need to be running.

In [ ]:
import os
import json
import sqlite3

import pandas as pd

# resolve the project root (folder containing product_run.py)
_here = os.getcwd()
while not os.path.exists(os.path.join(_here, "product_run.py")) and os.path.dirname(_here) != _here:
    _here = os.path.dirname(_here)
PROJECT_ROOT = _here

DB_PATH = os.path.join(PROJECT_ROOT, os.environ.get("PRODUCT_DB_NAME", "product_database.db"))
EXPORT_DIR = os.path.join(PROJECT_ROOT, "product_export")

print("database :", DB_PATH, "| exists:", os.path.exists(DB_PATH))
print("export to:", EXPORT_DIR)

## 1. Products available to export (top level)

In [ ]:
with sqlite3.connect(DB_PATH) as conn:
    top_level = pd.read_sql_query(
        "SELECT primary_product_key, product_type, description, child_count, created_at "
        "FROM product_latest WHERE parent_product_key IS NULL ORDER BY id",
        conn,
    )
top_level

## 2. Choose what to export

Put one or more `PrimaryProductKey` values in `KEYS`. Leave it empty to export **every**
top-level product.

In [ ]:
KEYS = [
    # "urn:inexto:id:prd:tobacco:pmi:favp:FA003115.00.3",
    # "urn:inexto:id:prd:tobacco:pmi:favp:DK001129.00.3",
]

if not KEYS:
    KEYS = top_level["primary_product_key"].tolist()

print(f"{len(KEYS)} product(s) to export")

## 3. Export

Each product is rebuilt from its stored `product_json`, and its children are put back
into `ChildProducts` by following `parent_product_key`. The file name is the last part
of the key, e.g. `FA003115.00.3.json`.

In [ ]:
def load_products():
    """Every product in product_latest, plus who is inside whom."""
    with sqlite3.connect(DB_PATH) as conn:
        rows = conn.execute(
            "SELECT primary_product_key, parent_product_key, product_json "
            "FROM product_latest ORDER BY id"
        ).fetchall()
    product_json = {key: data for key, _parent, data in rows}
    children_of = {}
    for key, parent, _data in rows:
        if parent is not None:
            children_of.setdefault(parent, []).append(key)
    return product_json, children_of


def build_product(key, product_json, children_of):
    """One product with its ChildProducts nested again, as in the source file."""
    product = json.loads(product_json[key])
    children = children_of.get(key, [])
    product["ChildProducts"] = [
        build_product(child, product_json, children_of) for child in children
    ] or None
    return product


def export_products(keys, export_dir=EXPORT_DIR):
    os.makedirs(export_dir, exist_ok=True)
    product_json, children_of = load_products()
    written = []
    for key in keys:
        if key not in product_json:
            print("not found, skipped:", key)
            continue
        product = build_product(key, product_json, children_of)
        file_name = key.rsplit(":", 1)[-1] + ".json"
        path = os.path.join(export_dir, file_name)
        with open(path, "w", encoding="utf-8") as f:
            json.dump(product, f, indent="\t", ensure_ascii=False)
        written.append({"primary_product_key": key, "file": file_name})
    return pd.DataFrame(written)


exported = export_products(KEYS)
exported

## 4. (Optional) Check against the original files

For every exported product that also has a source file in `product_data/`, compare the data.
`True` means every field and value matches (formatting such as indentation may differ).

In [ ]:
originals = {}
for name in os.listdir(os.path.join(PROJECT_ROOT, "product_data")):
    if name.endswith(".json"):
        with open(os.path.join(PROJECT_ROOT, "product_data", name), "rb") as f:
            data = json.loads(f.read())
        originals[data["PrimaryProductKey"]] = name

checks = []
for row in exported.itertuples():
    if row.primary_product_key not in originals:
        continue
    with open(os.path.join(PROJECT_ROOT, "product_data", originals[row.primary_product_key]), "rb") as f:
        original = json.loads(f.read())
    with open(os.path.join(EXPORT_DIR, row.file), "rb") as f:
        rebuilt = json.loads(f.read())
    checks.append({"file": row.file, "original": originals[row.primary_product_key],
                   "same_data": rebuilt == original})
pd.DataFrame(checks)